# Get **Dataset**

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("carlosrunner/pizza-not-pizza")

print("Path to dataset files:", path)

100%|██████████| 101M/101M [00:01<00:00, 93.2MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/carlosrunner/pizza-not-pizza/versions/1


# Import Libaraies

In [31]:
import tensorflow as tf
import cv2
import os
import random
import numpy as np
from sklearn.model_selection import train_test_split

# OS

In [5]:
print(path)

/root/.cache/kagglehub/datasets/carlosrunner/pizza-not-pizza/versions/1


In [ ]:
os.listdir('/root/.cache/kagglehub/datasets/carlosrunner/pizza-not-pizza/versions/1/pizza_not_pizza')

In [9]:
os.listdir(os.path.join(path, "pizza_not_pizza"))

['pizza', 'food101_subset.py', 'not_pizza']

In [10]:
BASE_DIR=os.path.join(path, "pizza_not_pizza")

In [11]:
pizza_dir=os.path.join(BASE_DIR, "pizza")
not_pizza_dir=os.path.join(BASE_DIR, "not_pizza")

In [18]:
print(len(os.listdir(pizza_dir)))
print(len(os.listdir(not_pizza_dir)))

983
983


# CV

In [25]:
Image_Size=(128,128)

In [28]:
def get_data_and_labels(data_dir,label):
    data=[]
    for img in os.listdir(data_dir):
        path=os.path.join(data_dir, img)
        img_arr=cv2.imread(path)
        img_arr=cv2.resize(img_arr, Image_Size)
        data.append([img_arr,label])
    return data

In [30]:
pizza_data=get_data_and_labels(pizza_dir,1)
not_pizza_data=get_data_and_labels(not_pizza_dir,0)
all_data=pizza_data+not_pizza_data

# Shuffle The DataSet

In [32]:
random.shuffle(all_data)

# Split img & Labels

In [34]:
images = []
labels = []

for image, label in all_data:
    images.append(image)
    labels.append(label)

In [36]:
images = np.array(images)
labels = np.array(labels)

In [37]:
print(images.shape)
print(labels.shape)

(1966, 128, 128, 3)
(1966,)


In [38]:
images=images.astype('float32')/255.0
print(images.min())
print(images.max())

0.0
1.0


# Train Test Split

In [41]:
x_train , x_test , y_train , y_test = train_test_split(images,labels,test_size=0.2,random_state=42)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(1572, 128, 128, 3)
(394, 128, 128, 3)
(1572,)
(394,)


# MLP MODEL

In [45]:
model=tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(128,128,3)),
    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dense(64,activation='relu'),
    tf.keras.layers.Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [47]:
model_compile=model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [48]:
batch_size=32
validation_split=0.2
epochs=10
Model_Learn=model.fit(
    x_train,y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=validation_split
)

Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 97ms/step - accuracy: 0.5680 - loss: 2.1438 - val_accuracy: 0.5873 - val_loss: 1.3229
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 112ms/step - accuracy: 0.6189 - loss: 1.0210 - val_accuracy: 0.5397 - val_loss: 1.4955
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 85ms/step - accuracy: 0.5927 - loss: 1.4010 - val_accuracy: 0.6444 - val_loss: 0.7609
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - accuracy: 0.6738 - loss: 0.6439 - val_accuracy: 0.6381 - val_loss: 0.6672
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 114ms/step - accuracy: 0.6921 - loss: 0.6150 - val_accuracy: 0.5714 - val_loss: 0.8462
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 85ms/step - accuracy: 0.7446 - loss: 0.5074 - val_accuracy: 0.6254 - val_loss: 0.6965
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - accuracy: 0.7295 - loss: 0.5996 - val_accuracy: 0.5905 - val_loss: 0.8464
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 109ms/step - accuracy: 0.6961 - loss: 0.6536 - val_accuracy: 0.5746 

In [50]:
test_accuracy=model.evaluate(x_test,y_test)
print("Test Accuracy:", test_accuracy )

13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.6218 - loss: 0.8018
Test Accuracy: [0.8017787933349609, 0.6218274235725403]


In [60]:
model.save('mlp_model.keras')
print(os.path.exists('mlp_model.keras'))
from google.colab import files

files.download('mlp_model.keras')

True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>